# Shared definitions

This notebook defines every shared function for the retinal adaptation study.
It runs nothing. Each phase notebook executes it first with `%run definitions.ipynb`.

Rules for this file:

- Keep it fixed while model runs execute (TRADEOFFS.md limit 7).
- Each run records the SHA-256 of this file.
- Phase 1 appends the model, training, metric, and bootstrap definitions in
  clearly marked cells below the Phase 0 sections.

Registered protocol: `retinal_adaptation_study_v1.md`.


In [ ]:
# Package pins. Run once per machine, then restart the kernel if it installed anything.
# The cell only installs packages whose installed version differs from the pin.
# It never installs or replaces torch: training machines keep their CUDA build.
import importlib.metadata
import subprocess
import sys

REQUIRED_PINS = {
    "numpy": "2.2.6",
    "pandas": "2.3.3",
    "pillow": "12.0.0",
    "opencv-python-headless": "4.12.0.88",
    "scikit-learn": "1.7.2",
    "matplotlib": "3.10.7",
    "nbformat": "5.10.4",
}
TRAINING_MINIMUM = {"torch": "2.8.0", "torchvision": "0.23.0"}


def _installed(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


_todo = [f"{p}=={v}" for p, v in REQUIRED_PINS.items() if _installed(p) != v]
if _todo:
    print("installing:", " ".join(_todo))
    _cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + _todo
    _done = subprocess.run(_cmd, capture_output=True, text=True)
    if _done.returncode != 0 and "externally-managed-environment" in _done.stderr:
        # Debian-managed interpreters (the Runpod PyTorch image) refuse plain
        # installs. The pins must reach the same interpreter that runs the
        # notebook, so install into it explicitly.
        print("externally managed environment: installing with --break-system-packages")
        _done = subprocess.run(_cmd + ["--break-system-packages"], capture_output=True, text=True)
    if _done.returncode != 0:
        print(_done.stdout[-2000:])
        print(_done.stderr[-2000:])
        raise RuntimeError("the pinned install failed; read the output above.")
    print("install finished. Restart the kernel, then run this notebook again.")
else:
    print("required packages already match the pins.")

for _p, _minimum in TRAINING_MINIMUM.items():
    _v = _installed(_p)
    if _v is None:
        print(f"note: {_p} is absent. Training phases need {_p}>={_minimum}; data phases do not.")
    else:
        print(f"{_p} {_v} present (training phases need >= {_minimum}).")


In [ ]:
import hashlib
import io
import json
import os
import platform
import random
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import importlib.metadata as _ilm

_REQUIRED_PINS = {
    "numpy": "2.2.6",
    "pandas": "2.3.3",
    "pillow": "12.0.0",
    "opencv-python-headless": "4.12.0.88",
}
for _p, _v in _REQUIRED_PINS.items():
    _have = _ilm.version(_p)
    if _have != _v:
        raise RuntimeError(
            f"{_p} is {_have}, the pin is {_v}. Run the install cell, restart the kernel, rerun."
        )

try:
    import torch  # noqa: F401
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False


class PhaseStop(Exception):
    """Raised when a registered check fails and the phase must stop for review."""


print(f"python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} "
      f"| pillow {_ilm.version('pillow')} | opencv {cv2.__version__} | torch available: {TORCH_AVAILABLE}")


In [ ]:
# Registered constants (protocol Appendix B plus fixed implementation choices).

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "images").is_dir():
    raise PhaseStop(f"no images/ directory under {PROJECT_ROOT}. Launch Jupyter from the project root.")

IMAGES_DIR = PROJECT_ROOT / "images"
PHASE0_SOURCE = PROJECT_ROOT / "phase0" / "source"
PHASE0_ARTIFACTS = PROJECT_ROOT / "phase0" / "artifacts"
PHASE0_REPORTS = PROJECT_ROOT / "phase0" / "reports"

# Label schema of the merged MuReD table: ID column plus 20 labels.
ALL_LABELS = ["DR", "NORMAL", "MH", "ODC", "TSLN", "ARMD", "DN", "MYA", "BRVO", "ODP",
              "CRVO", "CNV", "RS", "ODE", "LS", "CSR", "HTR", "ASR", "CRS", "OTHER"]
TARGETS = ["DR", "NORMAL", "ODC", "MH", "DN"]
SOURCES = ["aria", "rfmid", "stare"]

# Registered SHA-256 of the label tables downloaded from doi:10.17632/pc4mb3h8hz.1.
LABEL_TABLE_SHA256 = {
    "train_data.csv": "a40f340cba6dd7141a3194214434b51484186dec8903e25d096853b995a5a7ee",
    "val_data.csv": "d7967f05e7f422002e071cdd5288cf822185ac86ad9f237fe698b29915223fb7",
}

EXPECTED_TABLE_ROWS = 2208
EXPECTED_IMAGE_FILES = 2451
EXPECTED_COHORT = 1489

N_FOLDS = 5
# Outer test capacities by fold id 1..5. Fold 5 holds 297 by registered convention.
OUTER_TEST_CAPACITY = {1: 298, 2: 298, 3: 298, 4: 298, 5: 297}
FIT_IMAGES_PER_FOLD = 1044

SPLIT_SEED = 2026
BOOTSTRAP_SEED = 2026
BOOTSTRAP_REPLICATES = 10000

SUBSET_SIZES = [50, 100, 251, 501, 1044]
SEED_SUBSET_SIZES = {42: [50, 100, 251, 501, 1044], 43: [50, 100], 44: [50]}
TRAIN_SEEDS = [42, 43, 44]

# Preprocessing (protocol section 2.4) plus fixed implementation choices:
# center indices use floor division, the scan comparison is strictly greater,
# square padding puts the odd pixel on the right or bottom edge.
IMG_SIZE = 224
FOV_THRESHOLD_FRACTION = 0.06
PREPROCESSING_VERSION = "fov-crop-224-v1"
ROTATION_DEGREES = 10.0
HFLIP_PROBABILITY = 0.5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Duplicate audit, stage 1 screen: 63-bit DCT hash (8x8 low-frequency block minus
# the DC term, median threshold) on the 224 crop. Candidate when Hamming <= 8.
PHASH_HAMMING_MAX = 8

# Duplicate audit, stage 2 verification: ORB keypoints on the original image
# (longest side scaled to 768, CLAHE on the green channel, keypoints masked to
# the eroded retina) matched with cross-check, then a seeded RANSAC similarity
# fit at 3 px. Validation on this cohort: same-eye pairs scored 57 to 870
# inliers, distinct-eye pairs scored 0 to 17, one pair landed between.
HIRES_SIDE = 768
HIRES_ORB_FEATURES = 4000
HIRES_FAST_THRESHOLD = 10
HIRES_CLAHE_CLIP = 3.0
HIRES_CLAHE_GRID = 8
HIRES_MASK_LUMINANCE = 10
HIRES_MASK_ERODE = 31
HIRES_RANSAC_PX = 3.0
HIRES_RANSAC_ITERS = 5000
HIRES_RANSAC_CONFIDENCE = 0.999
HIRES_RNG_SEED = 2026
HIRES_CONFIRM_INLIERS = 40
HIRES_REVIEW_INLIERS = 18

# Model and training constants, registered now, used from Phase 1 onward.
BACKBONE_WEIGHTS = "IMAGENET1K_V2"
DKA_BOTTLENECK = 16
DKA_KERNELS = (51, 5)
STANDARD_ADAPTER_WIDTH = 37
NECK_WIDTH = 256
NECK_BLOCKS = 4
PRIMARY_MLP_WIDTH = 64
CAPACITY_MLP_WIDTH = 112
MHC_STREAMS = 4
SINKHORN_ITERATIONS = 20
FINETUNE_EPOCHS = 100
WARMUP_MAX_EPOCHS = 15
WARMUP_PATIENCE = 3
BATCH_SIZE = 32
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0
LR_MODULES = 1e-3
LR_HEAD = 1e-4
NUM_WORKERS = 4
KERNEL_STUDY_LARGE = [7, 15, 31, 51]

for _d in [PHASE0_ARTIFACTS, PHASE0_REPORTS]:
    _d.mkdir(parents=True, exist_ok=True)


In [ ]:
# Hashing and provenance helpers.

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def definitions_sha256():
    return sha256_file(PROJECT_ROOT / "definitions.ipynb")


def package_versions():
    names = ["numpy", "pandas", "pillow", "opencv-python-headless",
             "scikit-learn", "matplotlib", "nbformat"]
    out = {}
    for n in names:
        try:
            out[n] = _ilm.version(n)
        except _ilm.PackageNotFoundError:
            out[n] = None
    if TORCH_AVAILABLE:
        out["torch"] = torch.__version__
        try:
            import torchvision
            out["torchvision"] = torchvision.__version__
        except ImportError:
            out["torchvision"] = None
    return out


def environment_record():
    rec = {
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "pythonhashseed": os.environ.get("PYTHONHASHSEED"),
        "packages": package_versions(),
        "definitions_sha256": definitions_sha256(),
    }
    if TORCH_AVAILABLE and torch.cuda.is_available():
        rec["gpu"] = torch.cuda.get_device_name(0)
    return rec


def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str)
    return sha256_file(path)


In [ ]:
# Determinism helpers.
# CUBLAS_WORKSPACE_CONFIG must exist before the first CUDA context. Setting it
# here covers every phase because each phase runs this notebook first.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def enable_torch_determinism():
    if not TORCH_AVAILABLE:
        raise PhaseStop("torch is absent on this machine.")
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


def require_pythonhashseed():
    v = os.environ.get("PYTHONHASHSEED")
    if v != "2026":
        raise PhaseStop(
            f"PYTHONHASHSEED is {v!r}, the protocol requires '2026' before Python starts. "
            "Relaunch Jupyter as RUNBOOK.md shows."
        )


In [ ]:
# Image preprocessing (protocol section 2.4).

def decode_rgb(image_bytes):
    img = Image.open(io.BytesIO(image_bytes))
    return np.asarray(img.convert("RGB"), dtype=np.uint8)


def fov_scan(arr):
    """Field-of-view scan. Returns a dict; ok=False marks the image for source review."""
    red = arr[:, :, 0].astype(np.float64)
    max_intensity = float(red.max())
    threshold = FOV_THRESHOLD_FRACTION * max_intensity
    h, w = red.shape
    row_hits = np.nonzero(red[h // 2, :] > threshold)[0]
    col_hits = np.nonzero(red[:, w // 2] > threshold)[0]
    if max_intensity <= 0 or row_hits.size == 0 or col_hits.size == 0:
        return {"ok": False, "max_intensity": max_intensity, "threshold": threshold,
                "height": h, "width": w,
                "left": -1, "right": -1, "top": -1, "bottom": -1}
    return {"ok": True, "max_intensity": max_intensity, "threshold": threshold,
            "height": h, "width": w,
            "left": int(row_hits[0]), "right": int(row_hits[-1]),
            "top": int(col_hits[0]), "bottom": int(col_hits[-1])}


def fov_crop_224(arr, scan):
    """Crop the scanned rectangle, pad to a centered black square, resize to 224."""
    crop = arr[scan["top"]:scan["bottom"] + 1, scan["left"]:scan["right"] + 1]
    h, w = crop.shape[:2]
    side = max(h, w)
    top_pad = (side - h) // 2
    left_pad = (side - w) // 2
    canvas = np.zeros((side, side, 3), dtype=np.uint8)
    canvas[top_pad:top_pad + h, left_pad:left_pad + w] = crop
    return Image.fromarray(canvas).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)


In [ ]:
# 63-bit perceptual hash, the duplicate-audit screen.

_DCT32 = None


def _dct_matrix(n=32):
    global _DCT32
    if _DCT32 is None:
        k = np.arange(n, dtype=np.float64).reshape(-1, 1)
        i = np.arange(n, dtype=np.float64).reshape(1, -1)
        m = np.cos(np.pi * (2.0 * i + 1.0) * k / (2.0 * n)) * np.sqrt(2.0 / n)
        m[0, :] = np.sqrt(1.0 / n)
        _DCT32 = m
    return _DCT32


def phash63(pil_image):
    gray = np.asarray(pil_image.convert("L").resize((32, 32), Image.BILINEAR), dtype=np.float64)
    d = _dct_matrix()
    freq = d @ gray @ d.T
    block = freq[:8, :8].flatten()[1:]
    median = float(np.median(block))
    bits = 0
    for value in block:
        bits = (bits << 1) | (1 if value > median else 0)
    return bits


In [ ]:
# Duplicate-audit verification: geometric matching on the original images.
# Two captures of one eye align on fine vessel detail and give a large seeded
# RANSAC consensus. Two different eyes with similar layout give a small one.
# Cohort validation: true pairs 57 to 870 inliers, false pairs 0 to 17.

class HiresVerifier:
    def __init__(self, stem_to_file, images_dir=None):
        self.stem_to_file = stem_to_file
        self.images_dir = Path(images_dir) if images_dir is not None else IMAGES_DIR
        self._clahe = cv2.createCLAHE(clipLimit=HIRES_CLAHE_CLIP,
                                      tileGridSize=(HIRES_CLAHE_GRID, HIRES_CLAHE_GRID))
        self._orb = cv2.ORB_create(nfeatures=HIRES_ORB_FEATURES, fastThreshold=HIRES_FAST_THRESHOLD)
        self._matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        self._cache = {}

    def _features(self, image_id):
        if image_id in self._cache:
            return self._cache[image_id]
        img = cv2.imread(str(self.images_dir / self.stem_to_file[image_id]))
        if img is None:
            raise PhaseStop(f"cv2 cannot read the image for {image_id}.")
        h, w = img.shape[:2]
        s = HIRES_SIDE / max(h, w)
        img = cv2.resize(img, (int(round(w * s)), int(round(h * s))), interpolation=cv2.INTER_AREA)
        g = self._clahe.apply(img[:, :, 1])
        lum = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        mask = cv2.erode((lum > HIRES_MASK_LUMINANCE).astype(np.uint8) * 255,
                         np.ones((HIRES_MASK_ERODE, HIRES_MASK_ERODE), np.uint8))
        kp, desc = self._orb.detectAndCompute(g, mask=mask)
        self._cache[image_id] = (kp, desc)
        return self._cache[image_id]

    def inliers(self, id_a, id_b):
        kpa, da = self._features(id_a)
        kpb, db = self._features(id_b)
        if da is None or db is None or len(kpa) < 8 or len(kpb) < 8:
            return 0
        matches = self._matcher.match(da, db)
        if len(matches) < 8:
            return 0
        src = np.float32([kpa[m.queryIdx].pt for m in matches])
        dst = np.float32([kpb[m.trainIdx].pt for m in matches])
        cv2.setRNGSeed(HIRES_RNG_SEED)
        M, inlier_mask = cv2.estimateAffinePartial2D(
            src, dst, method=cv2.RANSAC, ransacReprojThreshold=HIRES_RANSAC_PX,
            maxIters=HIRES_RANSAC_ITERS, confidence=HIRES_RANSAC_CONFIDENCE)
        if M is None:
            return 0
        return int(inlier_mask.sum())


def duplicate_rule(hires_inliers):
    if hires_inliers >= HIRES_CONFIRM_INLIERS:
        return "confirm"
    if hires_inliers >= HIRES_REVIEW_INLIERS:
        return "review"
    return "reject"


In [ ]:
# Cohort construction helpers.

_ARIA_PREFIX = re.compile(r"^\(\d+\)")
_ARIA_SUBJECT = re.compile(r"^(aria_[a-z]+_\d+)")


def image_source(image_id):
    stem = _ARIA_PREFIX.sub("", image_id)
    if stem.startswith("aria"):
        return "aria"
    if stem.startswith("im"):
        return "stare"
    if stem.isdigit():
        return "rfmid"
    raise PhaseStop(f"image id {image_id!r} matches no known source pattern.")


def source_subject(image_id):
    """ARIA ids encode a subject; STARE and RFMiD ids do not."""
    stem = _ARIA_PREFIX.sub("", image_id)
    m = _ARIA_SUBJECT.match(stem)
    if m:
        return m.group(1)
    return ""


def base_group_key(image_id):
    subject = source_subject(image_id)
    if subject:
        return f"{image_source(image_id)}:{subject}"
    return f"{image_source(image_id)}:{image_id}"


def load_label_tables():
    frames = []
    for name in ["train_data.csv", "val_data.csv"]:
        path = PHASE0_SOURCE / name
        digest = sha256_file(path)
        if digest != LABEL_TABLE_SHA256[name]:
            raise PhaseStop(f"{name} hash {digest} differs from the registered value.")
        frames.append(pd.read_csv(path, dtype={"ID": str}))
    table = pd.concat(frames, ignore_index=True)
    if list(table.columns) != ["ID"] + ALL_LABELS:
        raise PhaseStop(f"unexpected label columns: {list(table.columns)}")
    if len(table) != EXPECTED_TABLE_ROWS or table["ID"].nunique() != EXPECTED_TABLE_ROWS:
        raise PhaseStop(f"merged table has {len(table)} rows, expected {EXPECTED_TABLE_ROWS} unique.")
    if not table[ALL_LABELS].isin([0, 1]).all().all():
        raise PhaseStop("a label value outside {0, 1} exists in the merged table.")
    return table


class DisjointSets:
    def __init__(self, items):
        self.parent = {i: i for i in items}

    def find(self, a):
        while self.parent[a] != a:
            self.parent[a] = self.parent[self.parent[a]]
            a = self.parent[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            if rb < ra:
                ra, rb = rb, ra
            self.parent[rb] = ra


In [ ]:
# Grouped iterative stratification with exact bin capacities.
# Units are analysis groups. Balancing columns are the five targets plus the
# three source indicators. Scarcest label first; each group goes to the bin
# with the largest unmet need for that label among bins with room; ties fall
# to the larger remaining capacity, then to a draw from the SPLIT_SEED stream.

def grouped_stratified_partition(groups, capacities, rng):
    """groups: list of dicts with gid, size, counts (numpy vector).
    capacities: list of bin sizes summing to the total image count.
    Returns {gid: bin_index}."""
    n_bins = len(capacities)
    total = sum(g["size"] for g in groups)
    if total != sum(capacities):
        raise PhaseStop(f"capacities sum to {sum(capacities)}, images sum to {total}.")
    totals = np.sum([g["counts"] for g in groups], axis=0).astype(np.float64)
    n_cols = totals.shape[0]
    desired = np.outer(np.asarray(capacities, dtype=np.float64), totals) / float(total)
    assigned = np.zeros((n_bins, n_cols))
    room = list(capacities)
    pending = {g["gid"]: g for g in groups}
    remaining_totals = totals.copy()
    assignment = {}
    while pending:
        active = [k for k in range(n_cols) if remaining_totals[k] > 0]
        k_star = None
        if active:
            k_star = min(active, key=lambda k: (remaining_totals[k], k))
            candidates = [g for g in pending.values() if g["counts"][k_star] > 0]
            if not candidates:
                remaining_totals[k_star] = 0
                continue
        else:
            candidates = list(pending.values())
        if k_star is None:
            candidates.sort(key=lambda g: (-g["size"], g["gid"]))
        else:
            candidates.sort(key=lambda g: (-g["counts"][k_star], -g["size"], g["gid"]))
        g = candidates[0]
        feasible = [b for b in range(n_bins) if room[b] >= g["size"]]
        if not feasible:
            raise PhaseStop(f"no bin has room for group {g['gid']} of size {g['size']}.")
        if k_star is None:
            need = [float(room[b]) for b in feasible]
        else:
            need = [float(desired[b, k_star] - assigned[b, k_star]) for b in feasible]
        top = max(need)
        best = [b for b, nd in zip(feasible, need) if nd == top]
        if len(best) > 1:
            most_room = max(room[b] for b in best)
            best = [b for b in best if room[b] == most_room]
        pick = best[0] if len(best) == 1 else int(rng.choice(np.array(sorted(best))))
        assignment[g["gid"]] = pick
        assigned[pick] += g["counts"]
        room[pick] -= g["size"]
        remaining_totals -= g["counts"]
        del pending[g["gid"]]
    if any(r != 0 for r in room):
        raise PhaseStop(f"capacities not met, leftover room {room}.")
    return assignment


def build_group_units(frame):
    """One unit per analysis_group_id with image count and balancing counts."""
    units = []
    for gid, part in frame.groupby("analysis_group_id", sort=True):
        counts = [int(part[t].sum()) for t in TARGETS]
        counts += [int((part["source_dataset"] == s).sum()) for s in SOURCES]
        units.append({"gid": gid, "size": len(part), "counts": np.asarray(counts, dtype=np.float64)})
    return units


def check_label_support(frame, name, require_negative=True):
    for t in TARGETS:
        positives = int(frame[t].sum())
        if positives == 0:
            raise PhaseStop(f"{name} has no positive {t}.")
        if require_negative and positives == len(frame):
            raise PhaseStop(f"{name} has no negative {t}.")


In [ ]:
# Nested training subsets (protocol section 2.3).
# Selection grows from 50 to 1044 inside one seeded draw per fold and seed, so
# each smaller subset sits inside the next larger one. Quotas per target follow
# fit-pool prevalence. The scarcest deficit fills first; a tie falls to the
# rarer label, then to label order. Every draw comes from one seeded stream.

def nested_subsets(fit_frame, fold, train_seed):
    sizes = SEED_SUBSET_SIZES[train_seed]
    rng = np.random.default_rng([SPLIT_SEED, fold, train_seed])
    ids = list(fit_frame["image_id"])
    labels = {row.image_id: {t: int(getattr(row, t)) for t in TARGETS}
              for row in fit_frame.itertuples()}
    n_pool = len(ids)
    prevalence = {t: float(fit_frame[t].sum()) / n_pool for t in TARGETS}
    scarcity_order = sorted(TARGETS, key=lambda t: (prevalence[t], TARGETS.index(t)))
    selected = []
    remaining = sorted(ids)
    out = {}
    for n in sorted(sizes):
        if n == n_pool:
            out[n] = sorted(ids)
            continue
        quotas = {t: max(1, round(prevalence[t] * n)) for t in TARGETS}
        while len(selected) < n:
            counts = {t: sum(labels[i][t] for i in selected) for t in TARGETS}
            deficits = {t: quotas[t] - counts[t] for t in TARGETS}
            pool = []
            for t in sorted(TARGETS, key=lambda x: (-deficits[x], scarcity_order.index(x))):
                if deficits[t] <= 0:
                    continue
                pool = [i for i in remaining if labels[i][t] == 1]
                if pool:
                    break
            if not pool:
                pool = remaining
            pick = pool[int(rng.integers(len(pool)))]
            selected.append(pick)
            remaining.remove(pick)
        subset = sorted(selected)
        for t in TARGETS:
            positives = sum(labels[i][t] for i in subset)
            if positives == 0 or positives == len(subset):
                raise PhaseStop(f"fold {fold} seed {train_seed} n={n}: {t} lacks both label values.")
        out[n] = subset
    return out


## Phase 1 sections

The cells below define the data pipeline, models, training, metrics, run
controller, and bootstrap for Phases 1 through 4. Protocol sections 4 through
7 fix every value. Registered implementation choices that the protocol leaves
open are stated in the comments of each cell.


In [ ]:
# ---------------------------------------------------------------------------
# Data pipeline. Workers only read RAM; every random draw comes from a seeded
# generator keyed by (fold, train seed, n, stage, epoch), so matched cells see
# identical image orders and augmentation draws regardless of epoch counts.
# ---------------------------------------------------------------------------
if TORCH_AVAILABLE:
    import torch.nn as nn
    from torchvision.models import resnet50, ResNet50_Weights
    from torchvision.transforms.v2 import functional as TVF
    from torchvision.transforms import InterpolationMode


def stable_seed(*parts):
    digest = hashlib.sha256("|".join(str(p) for p in parts).encode()).digest()
    return int.from_bytes(digest[:8], "big") % (2**63 - 1)


def load_cached_uint8(image_ids):
    """Stack the cached 224 crops as one uint8 tensor (N, 3, 224, 224)."""
    out = np.zeros((len(image_ids), 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for k, i in enumerate(image_ids):
        a = np.asarray(Image.open(PHASE0_ARTIFACTS / "image_cache" / f"{i}.png").convert("RGB"),
                       dtype=np.uint8)
        out[k] = a.transpose(2, 0, 1)
    return torch.from_numpy(out)


def epoch_order(n_images, fold, train_seed, n, stage, epoch):
    g = torch.Generator().manual_seed(stable_seed("order", fold, train_seed, n, stage, epoch))
    return torch.randperm(n_images, generator=g).tolist()


def epoch_augmentation_draws(n_images, fold, train_seed, n, stage, epoch):
    """One flip flag and one rotation angle per epoch position."""
    g = torch.Generator().manual_seed(stable_seed("aug", fold, train_seed, n, stage, epoch))
    flips = torch.rand(n_images, generator=g) < HFLIP_PROBABILITY
    angles = torch.rand(n_images, generator=g) * (2 * ROTATION_DEGREES) - ROTATION_DEGREES
    return flips, angles


_NORM_MEAN = None
_NORM_STD = None


def normalize_batch(x_float, device):
    global _NORM_MEAN, _NORM_STD
    if _NORM_MEAN is None or _NORM_MEAN.device != x_float.device:
        _NORM_MEAN = torch.tensor(IMAGENET_MEAN, device=x_float.device).view(1, 3, 1, 1)
        _NORM_STD = torch.tensor(IMAGENET_STD, device=x_float.device).view(1, 3, 1, 1)
    return (x_float - _NORM_MEAN) / _NORM_STD


def augment_and_normalize(x_uint8, positions, flips, angles, device, train):
    x = x_uint8.to(device, non_blocking=True).float() / 255.0
    if train:
        for j, pos in enumerate(positions):
            if bool(flips[pos]):
                x[j] = torch.flip(x[j], dims=[2])
            angle = float(angles[pos])
            x[j] = TVF.rotate(x[j], angle, interpolation=InterpolationMode.BILINEAR, fill=0.0)
    return normalize_batch(x, device)


In [ ]:
# ---------------------------------------------------------------------------
# Models (protocol section 4). Registered parameter counts:
# standard adapter 76,837 | DKA 75,856 | residual-64 neck 134,400 |
# mHC neck 232,812 | capacity-112 neck 232,896.
# ---------------------------------------------------------------------------
REGISTERED_PARAM_COUNTS = {
    "standard_adapter": 76837,
    "dka": 75856,
    "residual_neck": 134400,
    "mhc_neck": 232812,
    "capacity_neck": 232896,
}

if TORCH_AVAILABLE:

    class StandardAdapter(nn.Module):
        def __init__(self):
            super().__init__()
            self.down = nn.Conv2d(1024, STANDARD_ADAPTER_WIDTH, 1, bias=True)
            self.act = nn.GELU()
            self.up = nn.Conv2d(STANDARD_ADAPTER_WIDTH, 1024, 1, bias=True)
            nn.init.zeros_(self.up.weight)
            nn.init.zeros_(self.up.bias)

        def forward(self, x):
            return x + self.up(self.act(self.down(x)))

    class DualKernelAdapter(nn.Module):
        def __init__(self, large_kernel=None):
            super().__init__()
            k = int(large_kernel if large_kernel is not None else DKA_KERNELS[0])
            self.down = nn.Conv2d(1024, DKA_BOTTLENECK, 1, bias=True)
            self.dw_large = nn.Conv2d(DKA_BOTTLENECK, DKA_BOTTLENECK, k, stride=1,
                                      padding=(k - 1) // 2, groups=DKA_BOTTLENECK, bias=True)
            self.dw_small = nn.Conv2d(DKA_BOTTLENECK, DKA_BOTTLENECK, DKA_KERNELS[1], stride=1,
                                      padding=(DKA_KERNELS[1] - 1) // 2, groups=DKA_BOTTLENECK,
                                      bias=True)
            self.act = nn.GELU()
            self.up = nn.Conv2d(DKA_BOTTLENECK, 1024, 1, bias=True)
            nn.init.zeros_(self.up.weight)
            nn.init.zeros_(self.up.bias)

        def forward(self, x):
            u = self.down(x)
            v = self.dw_large(u) + self.dw_small(u)
            return x + self.up(self.act(v))

    class FrozenBackbone(nn.Module):
        def __init__(self):
            super().__init__()
            m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
            self.conv1, self.bn1, self.relu, self.maxpool = m.conv1, m.bn1, m.relu, m.maxpool
            self.layer1, self.layer2, self.layer3, self.layer4 = m.layer1, m.layer2, m.layer3, m.layer4
            self.avgpool = m.avgpool
            for p in self.parameters():
                p.requires_grad_(False)
            super().train(False)

        def train(self, mode=True):
            return super().train(False)  # batch-norm statistics stay frozen

        def features_pre(self, x):
            x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
            return self.layer3(self.layer2(self.layer1(x)))

        def features_post(self, x):
            x = self.avgpool(self.layer4(x))
            return torch.flatten(x, 1)

    def make_common_stem():
        return nn.Sequential(nn.LayerNorm(2048), nn.Linear(2048, NECK_WIDTH), nn.GELU())

    class ResidualBlock(nn.Module):
        def __init__(self, hidden):
            super().__init__()
            self.ln = nn.LayerNorm(NECK_WIDTH)
            self.fc1 = nn.Linear(NECK_WIDTH, hidden)
            self.act = nn.GELU()
            self.fc2 = nn.Linear(hidden, NECK_WIDTH)
            nn.init.zeros_(self.fc2.weight)
            nn.init.zeros_(self.fc2.bias)

        def forward(self, z):
            return z + self.fc2(self.act(self.fc1(self.ln(z))))

    class ResidualNeck(nn.Module):
        def __init__(self, hidden):
            super().__init__()
            self.blocks = nn.ModuleList(ResidualBlock(hidden) for _ in range(NECK_BLOCKS))

        def forward(self, z):
            for b in self.blocks:
                z = b(z)
            return z

    def _rms_norm(x, eps=1e-6):
        return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)

    def sinkhorn_knopp(logits, iterations=SINKHORN_ITERATIONS):
        with torch.amp.autocast(device_type=logits.device.type, enabled=False):
            m = logits.float()
            m = m - m.amax(dim=(-2, -1), keepdim=True)
            k = torch.exp(m)
            for _ in range(iterations):
                k = k / k.sum(dim=-1, keepdim=True)
                k = k / k.sum(dim=-2, keepdim=True)
        return k

    class MHCBlock(nn.Module):
        def __init__(self):
            super().__init__()
            s = MHC_STREAMS
            self.ln = nn.LayerNorm(NECK_WIDTH)
            self.fc1 = nn.Linear(NECK_WIDTH, PRIMARY_MLP_WIDTH)
            self.act = nn.GELU()
            self.fc2 = nn.Linear(PRIMARY_MLP_WIDTH, NECK_WIDTH)
            nn.init.zeros_(self.fc2.weight)
            nn.init.zeros_(self.fc2.bias)
            self.proj = nn.Linear(s * NECK_WIDTH, s + s + s * s, bias=False)
            nn.init.xavier_uniform_(self.proj.weight)
            self.alpha_pre = nn.Parameter(torch.tensor(0.01))
            self.alpha_post = nn.Parameter(torch.tensor(0.01))
            self.alpha_res = nn.Parameter(torch.tensor(0.01))
            self.bias_pre = nn.Parameter(torch.full((s,), float(np.log(0.25 / 0.75))))
            self.bias_post = nn.Parameter(torch.zeros(s))
            target = 0.99 * np.eye(s) + 0.01 * np.ones((s, s)) / s
            self.bias_res = nn.Parameter(torch.tensor(np.log(target), dtype=torch.float32))

        def no_decay_parameters(self):
            return [self.alpha_pre, self.alpha_post, self.alpha_res,
                    self.bias_pre, self.bias_post, self.bias_res]

        def forward(self, X):
            s = MHC_STREAMS
            b = X.shape[0]
            flat = _rms_norm(X.reshape(b, s * NECK_WIDTH))
            logits = self.proj(flat)
            pre = torch.sigmoid(self.alpha_pre * logits[:, :s] + self.bias_pre)
            post = 2.0 * torch.sigmoid(self.alpha_post * logits[:, s:2 * s] + self.bias_post)
            res_logits = (self.alpha_res * logits[:, 2 * s:]).reshape(b, s, s) + self.bias_res
            H_res = sinkhorn_knopp(res_logits).to(X.dtype)
            u = torch.einsum("bs,bsd->bd", pre, X)
            r = self.fc2(self.act(self.fc1(self.ln(u))))
            return torch.bmm(H_res, X) + post.unsqueeze(-1) * r.unsqueeze(1)

    class MHCNeck(nn.Module):
        def __init__(self):
            super().__init__()
            self.blocks = nn.ModuleList(MHCBlock() for _ in range(NECK_BLOCKS))

        def no_decay_parameters(self):
            return [p for blk in self.blocks for p in blk.no_decay_parameters()]

        def forward(self, z):
            X = z.unsqueeze(1).repeat(1, MHC_STREAMS, 1)
            for b in self.blocks:
                X = b(X)
            return X.mean(dim=1)

    class CellModel(nn.Module):
        """Frozen backbone plus the trainable parts of one cell."""

        def __init__(self, backbone, adapter, stem, neck, head):
            super().__init__()
            self.backbone = backbone
            self.adapter = adapter
            self.stem = stem
            self.neck = neck
            self.head = head

        def forward(self, x):
            f = self.backbone.features_pre(x)
            if self.adapter is not None:
                f = self.adapter(f)
            f = self.backbone.features_post(f)
            if self.stem is not None:
                f = self.neck(self.stem(f))
            return self.head(f)

        def trainable_named_parameters(self):
            return [(n, p) for n, p in self.named_parameters()
                    if not n.startswith("backbone.")]


_BACKBONE = None


def get_backbone():
    global _BACKBONE
    if _BACKBONE is None:
        _BACKBONE = FrozenBackbone()
    return _BACKBONE


CELL_DESIGNS = {
    "A": {"adapter": "standard", "neck": "residual"},
    "B": {"adapter": "dka", "neck": "residual"},
    "C": {"adapter": "standard", "neck": "mhc"},
    "D": {"adapter": "dka", "neck": "mhc"},
    "E": {"adapter": "standard", "neck": "capacity"},
    "F": {"adapter": "dka", "neck": "capacity"},
}


def build_cell_model(cell, fold, train_seed, n, dka_large=None):
    """Matched stem and head come from a seed shared by every cell of one
    (fold, seed, n) set; adapter and neck come from a cell-specific seed."""
    backbone = get_backbone()
    if cell == "probe":
        torch.manual_seed(stable_seed("probe-head", fold, train_seed, n))
        head = nn.Linear(2048, len(TARGETS))
        return CellModel(backbone, None, None, None, head)
    design = CELL_DESIGNS[cell]
    torch.manual_seed(stable_seed("stem-head", fold, train_seed, n))
    stem = make_common_stem()
    head = nn.Linear(NECK_WIDTH, len(TARGETS))
    torch.manual_seed(stable_seed("cell", cell, fold, train_seed, n, dka_large))
    adapter = StandardAdapter() if design["adapter"] == "standard" else DualKernelAdapter(dka_large)
    if design["neck"] == "residual":
        neck = ResidualNeck(PRIMARY_MLP_WIDTH)
    elif design["neck"] == "capacity":
        neck = ResidualNeck(CAPACITY_MLP_WIDTH)
    else:
        neck = MHCNeck()
    return CellModel(backbone, adapter, stem, neck, head)


def count_parameters(module):
    return sum(p.numel() for p in module.parameters())


In [ ]:
# ---------------------------------------------------------------------------
# Registered model tests (protocol section 4.8). Every test raises on failure.
# ---------------------------------------------------------------------------

def run_registered_model_tests():
    if not TORCH_AVAILABLE:
        raise PhaseStop("torch is absent; the model tests need it.")
    results = []
    torch.manual_seed(0)
    x = torch.randn(2, 1024, 14, 14)

    std = StandardAdapter()
    dka = DualKernelAdapter()
    results.append(("1 adapter shapes equal input",
                    std(x).shape == x.shape and dka(x).shape == x.shape))
    with torch.no_grad():
        results.append(("2 adapters are identity at init",
                        float((std(x) - x).abs().max()) < 1e-6
                        and float((dka(x) - x).abs().max()) < 1e-6))

    model_a = build_cell_model("A", 1, 42, 1044)
    model_c = build_cell_model("C", 1, 42, 1044)
    z = torch.randn(3, NECK_WIDTH)
    with torch.no_grad():
        out_res = model_a.neck(z)
        out_mhc = model_c.neck(z)
    results.append(("3 necks equal at init", float((out_res - out_mhc).abs().max()) < 1e-6))

    logits = torch.randn(4, MHC_STREAMS, MHC_STREAMS)
    k = sinkhorn_knopp(logits)
    row_err = float((k.sum(-1) - 1).abs().max())
    col_err = float((k.sum(-2) - 1).abs().max())
    results.append(("4 sinkhorn sums within 1e-5", max(row_err, col_err) < 1e-5))

    model_d = build_cell_model("D", 1, 42, 1044)
    stats_before = [b.clone() for b in model_d.backbone.buffers()]
    model_d.train()
    img = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
    loss = model_d(img).sum()
    loss.backward()
    backbone_grads = [p.grad for _, p in model_d.backbone.named_parameters()]
    results.append(("5 no backbone gradient", all(g is None for g in backbone_grads)))
    stats_after = list(model_d.backbone.buffers())
    unchanged = all(torch.equal(a, b) for a, b in zip(stats_before, stats_after))
    results.append(("6 no batch-norm statistic change", unchanged))

    counts = {
        "standard_adapter": count_parameters(StandardAdapter()),
        "dka": count_parameters(DualKernelAdapter()),
        "residual_neck": count_parameters(ResidualNeck(PRIMARY_MLP_WIDTH)),
        "mhc_neck": count_parameters(MHCNeck()),
        "capacity_neck": count_parameters(ResidualNeck(CAPACITY_MLP_WIDTH)),
    }
    results.append(("7 registered parameter counts", counts == REGISTERED_PARAM_COUNTS))

    adapter_gap = abs(counts["standard_adapter"] - counts["dka"]) / counts["dka"]
    results.append(("8 adapter mismatch under 2%", adapter_gap < 0.02))
    neck_gap = abs(counts["capacity_neck"] - counts["mhc_neck"]) / counts["mhc_neck"]
    results.append(("9 capacity mismatch under 0.1%", neck_gap < 0.001))

    table = pd.DataFrame(results, columns=["test", "pass"])
    if not table["pass"].all():
        print(table.to_string(index=False))
        raise PhaseStop("a registered model test failed.")
    return table, counts


In [ ]:
# ---------------------------------------------------------------------------
# Metrics and threshold selection (protocol sections 5.5 and 7.1).
# ---------------------------------------------------------------------------

def fast_auc(y_true_bool, scores):
    """Rank-based AUC with average ranks for ties. NaN when one class is absent."""
    y = np.asarray(y_true_bool, dtype=bool)
    s = np.asarray(scores, dtype=np.float64)
    n_pos = int(y.sum())
    n_neg = y.size - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = np.argsort(s, kind="mergesort")
    ss = s[order]
    _, inverse, counts = np.unique(ss, return_inverse=True, return_counts=True)
    ends = np.cumsum(counts)
    starts = ends - counts
    avg_rank = (starts + 1 + ends) / 2.0
    ranks = np.empty(y.size, dtype=np.float64)
    ranks[order] = avg_rank[inverse]
    return float((ranks[y].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def macro_auc(labels, probs, class_indices=None):
    idx = range(labels.shape[1]) if class_indices is None else class_indices
    values = [fast_auc(labels[:, k] > 0.5, probs[:, k]) for k in idx]
    return float(np.mean(values)), values


def _f1_at(labels_bool, probs, threshold):
    pred = probs >= threshold
    tp = int(np.sum(pred & labels_bool))
    fp = int(np.sum(pred & ~labels_bool))
    fn = int(np.sum(~pred & labels_bool))
    denom = 2 * tp + fp + fn
    return 0.0 if denom == 0 else 2 * tp / denom


def select_thresholds(inner_probs, inner_labels):
    """Candidates are the unique inner-validation probabilities plus 0.5.
    Highest F1 wins; a tie takes the value closest to 0.5, then the higher one."""
    out = {}
    for k, name in enumerate(TARGETS):
        y = inner_labels[:, k] > 0.5
        candidates = np.unique(np.concatenate([inner_probs[:, k], [0.5]]))
        best = None
        for thr in candidates:
            score = _f1_at(y, inner_probs[:, k], thr)
            key = (score, -abs(thr - 0.5), thr)
            if best is None or key > best[0]:
                best = (key, thr)
        out[name] = float(best[1])
    return out


def thresholded_metrics(labels, probs, thresholds):
    thr = np.array([thresholds[t] for t in TARGETS])
    pred = probs >= thr
    y = labels > 0.5
    per_class_f1 = [_f1_at(y[:, k], probs[:, k], thr[k]) for k in range(len(TARGETS))]
    support = y.sum(axis=0)
    weighted_f1 = float(np.average(per_class_f1, weights=support)) if support.sum() else 0.0
    return {
        "macro_f1": float(np.mean(per_class_f1)),
        "weighted_f1": weighted_f1,
        "hamming_loss": float(np.mean(pred != y)),
        "exact_match": float(np.mean((pred == y).all(axis=1))),
        "per_class_f1": {t: float(v) for t, v in zip(TARGETS, per_class_f1)},
    }


In [ ]:
# ---------------------------------------------------------------------------
# Training (protocol section 5). Head-only warmup, then 100-epoch fine-tuning
# with cosine annealing. Checkpoint and warmup stop use inner-validation
# macro AUC; a tie keeps the earlier epoch (strict improvement rule).
# ---------------------------------------------------------------------------

def make_pos_weight(fit_labels):
    pos = fit_labels.sum(axis=0)
    neg = fit_labels.shape[0] - pos
    return torch.tensor(neg / pos, dtype=torch.float32)


def build_param_groups(model):
    """Adapter, stem, and neck train at 1e-3; the head at 1e-4. Weight decay
    1e-4 applies to convolution and linear weights; biases, normalization
    parameters, and router gates and biases decay zero."""
    no_decay_ids = set()
    for module in model.modules():
        if isinstance(module, nn.LayerNorm):
            no_decay_ids.update(id(p) for p in module.parameters())
    for part in [model.neck] if model.neck is not None else []:
        if hasattr(part, "no_decay_parameters"):
            no_decay_ids.update(id(p) for p in part.no_decay_parameters())
    groups = []
    parts = [("head", model.head, LR_HEAD)]
    for name in ["adapter", "stem", "neck"]:
        part = getattr(model, name)
        if part is not None:
            parts.append((name, part, LR_MODULES))
    for name, part, lr in parts:
        decay, no_decay = [], []
        for pname, p in part.named_parameters():
            if id(p) in no_decay_ids or pname.endswith("bias") or p.ndim <= 1:
                no_decay.append(p)
            else:
                decay.append(p)
        if decay:
            groups.append({"params": decay, "lr": lr, "weight_decay": WEIGHT_DECAY})
        if no_decay:
            groups.append({"params": no_decay, "lr": lr, "weight_decay": 0.0})
    return groups


def _clone_trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.trainable_named_parameters()}


def _load_trainable_state(model, state):
    with torch.no_grad():
        for n, p in model.trainable_named_parameters():
            p.copy_(state[n].to(p.device))


def evaluate_probabilities(model, images_uint8, device, amp, batch=64):
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, len(images_uint8), batch):
            x = augment_and_normalize(images_uint8[start:start + batch], None, None, None,
                                      device, train=False)
            with torch.amp.autocast(device_type=device, enabled=amp):
                logits = model(x)
            probs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(probs, axis=0)


def _train_one_epoch(model, optimizer, scaler, criterion, images, labels, loader_order,
                     flips, angles, device, amp):
    model.train()
    total, count = 0.0, 0
    for start in range(0, len(loader_order), BATCH_SIZE):
        positions = list(range(start, min(start + BATCH_SIZE, len(loader_order))))
        idx = [loader_order[p] for p in positions]
        x = augment_and_normalize(images[idx], positions, flips, angles, device, train=True)
        y = labels[idx].to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device, enabled=amp):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_((p for _, p in model.trainable_named_parameters()), GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total += float(loss.detach()) * len(idx)
        count += len(idx)
    return total / count


def _stage(model, stage_name, max_epochs, optimizer, scheduler, patience, criterion,
           data, device, amp, history, spec):
    """Runs one stage and returns (best_auc, best_epoch, best_state, epochs_run)."""
    images, labels, val_images, val_labels = data
    best_auc, best_epoch, best_state = -1.0, -1, None
    since_best = 0
    scaler = torch.amp.GradScaler(device, enabled=amp)
    for epoch in range(max_epochs):
        order = epoch_order(len(images), spec["fold"], spec["seed"], spec["n"], stage_name, epoch)
        flips, angles = epoch_augmentation_draws(len(images), spec["fold"], spec["seed"],
                                                 spec["n"], stage_name, epoch)
        loss = _train_one_epoch(model, optimizer, scaler, criterion,
                                images, labels, order, flips, angles, device, amp)
        val_probs = evaluate_probabilities(model, val_images, device, amp)
        aucs = [fast_auc(val_labels[:, k] > 0.5, val_probs[:, k]) for k in range(len(TARGETS))]
        val_auc = float(np.mean(aucs))
        history.append({"stage": stage_name, "epoch": epoch, "train_loss": loss,
                        "val_macro_auc": val_auc,
                        "lr": optimizer.param_groups[0]["lr"]})
        if val_auc > best_auc:
            best_auc, best_epoch, best_state = val_auc, epoch, _clone_trainable_state(model)
            since_best = 0
        else:
            since_best += 1
        if scheduler is not None:
            scheduler.step()
        if patience is not None and since_best >= patience:
            break
    return best_auc, best_epoch, best_state, len([h for h in history if h["stage"] == stage_name])


def train_run(spec, splits, dev_overrides=None):
    """Executes one registered run and returns its result payload.
    spec: cell, fold, seed, n, and for kernel runs dka_large.
    splits: dict with fit_ids, val_ids, test_ids (test_ids can be None),
    labels frame indexed by image_id, group map."""
    dev = dev_overrides or {}
    device = dev.get("device", "cuda")
    amp = dev.get("amp", device == "cuda")
    if device == "cuda" and not torch.cuda.is_available():
        raise PhaseStop("this run needs a CUDA device.")
    warmup_max = dev.get("warmup_max", WARMUP_MAX_EPOCHS)
    ft_epochs = dev.get("ft_epochs", FINETUNE_EPOCHS)

    set_all_seeds(spec["seed"])
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t_start = time.time()

    fit_ids, val_ids, test_ids = splits["fit_ids"], splits["val_ids"], splits.get("test_ids")
    label_of = splits["labels"]
    fit_labels_np = label_of.loc[fit_ids, TARGETS].to_numpy(dtype=np.float32)
    val_labels_np = label_of.loc[val_ids, TARGETS].to_numpy(dtype=np.float32)
    fit_images = load_cached_uint8(fit_ids)
    val_images = load_cached_uint8(val_ids)
    fit_labels = torch.from_numpy(fit_labels_np)

    model = build_cell_model(spec["cell"], spec["fold"], spec["seed"], spec["n"],
                             spec.get("dka_large")).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=make_pos_weight(fit_labels_np).to(device))
    history = []
    data = (fit_images, fit_labels, val_images, val_labels_np)

    head_params = list(model.head.parameters())
    warm_groups = [{"params": [p for p in head_params if p.ndim > 1],
                    "lr": 1e-3, "weight_decay": WEIGHT_DECAY},
                   {"params": [p for p in head_params if p.ndim <= 1],
                    "lr": 1e-3, "weight_decay": 0.0}]
    warm_opt = torch.optim.AdamW(warm_groups, betas=(0.9, 0.999), eps=1e-8)
    for name in ["adapter", "stem", "neck"]:
        part = getattr(model, name)
        if part is not None:
            for p in part.parameters():
                p.requires_grad_(False)
    w_auc, w_epoch, w_state, w_ran = _stage(model, "warmup", warmup_max, warm_opt, None,
                                            WARMUP_PATIENCE, criterion, data, device, amp,
                                            history, spec)
    _load_trainable_state(model, w_state)
    for name in ["adapter", "stem", "neck"]:
        part = getattr(model, name)
        if part is not None:
            for p in part.parameters():
                p.requires_grad_(True)

    if spec["cell"] == "probe":
        ft_groups = [{"params": [p for p in head_params if p.ndim > 1],
                      "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
                     {"params": [p for p in head_params if p.ndim <= 1],
                      "lr": LR_HEAD, "weight_decay": 0.0}]
    else:
        ft_groups = build_param_groups(model)
    ft_opt = torch.optim.AdamW(ft_groups, betas=(0.9, 0.999), eps=1e-8)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(ft_opt, T_max=ft_epochs, eta_min=0.0)
    f_auc, f_epoch, f_state, _ = _stage(model, "finetune", ft_epochs, ft_opt, scheduler, None,
                                        criterion, data, device, amp, history, spec)
    _load_trainable_state(model, f_state)

    inner_probs = evaluate_probabilities(model, val_images, device, amp)
    thresholds = select_thresholds(inner_probs, val_labels_np)
    result = {
        "history": pd.DataFrame(history),
        "state": _clone_trainable_state(model),
        "inner": {"image_ids": list(val_ids), "labels": val_labels_np, "probs": inner_probs},
        "thresholds": thresholds,
        "meta": {
            "cell": spec["cell"], "fold": spec["fold"], "seed": spec["seed"], "n": spec["n"],
            "dka_large": spec.get("dka_large"),
            "warmup_epochs_run": w_ran, "warmup_best_epoch": w_epoch,
            "finetune_best_epoch": f_epoch,
            "best_inner_macro_auc": f_auc,
            "trainable_parameters": sum(p.numel() for _, p in model.trainable_named_parameters()),
            "wall_seconds": round(time.time() - t_start, 1),
            "peak_gpu_bytes": int(torch.cuda.max_memory_allocated()) if device == "cuda" else 0,
            "device": device, "amp": bool(amp),
        },
    }
    if test_ids is not None:
        test_labels_np = label_of.loc[test_ids, TARGETS].to_numpy(dtype=np.float32)
        test_images = load_cached_uint8(test_ids)
        test_probs = evaluate_probabilities(model, test_images, device, amp)
        result["test"] = {"image_ids": list(test_ids), "labels": test_labels_np,
                          "probs": test_probs}
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return result


def run_mixed_precision_determinism_smoke_test():
    """One fp16 training step through the DKA-51 and mHC path, twice, on the
    target GPU. The two losses must match exactly under deterministic mode."""
    if not (TORCH_AVAILABLE and torch.cuda.is_available()):
        raise PhaseStop("the smoke test needs a CUDA device.")
    enable_torch_determinism()
    losses = []
    for _ in range(2):
        set_all_seeds(123)
        model = build_cell_model("D", 1, 42, 1044).to("cuda")
        opt = torch.optim.AdamW(build_param_groups(model))
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        x = torch.randn(4, 3, IMG_SIZE, IMG_SIZE, generator=torch.Generator().manual_seed(7))
        y = (torch.rand(4, len(TARGETS), generator=torch.Generator().manual_seed(8)) > 0.5).float()
        model.train()
        with torch.amp.autocast(device_type="cuda", enabled=True):
            loss = nn.BCEWithLogitsLoss()(model(x.to("cuda")), y.to("cuda"))
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_((p for _, p in model.trainable_named_parameters()), GRADIENT_CLIP)
        scaler.step(opt)
        scaler.update()
        losses.append(float(loss.detach()))
        del model
        torch.cuda.empty_cache()
    if losses[0] != losses[1]:
        raise PhaseStop(f"smoke test losses differ: {losses}")
    print(f"smoke test passed, loss {losses[0]:.6f} reproduced exactly")


In [ ]:
# ---------------------------------------------------------------------------
# Run manifest and controller. One process appends one manifest row per run;
# rerunning a notebook skips every run with a successful row.
# ---------------------------------------------------------------------------

def load_split_tables():
    cohort = pd.read_csv(PHASE0_ARTIFACTS / "cohort.csv", dtype={"image_id": str})
    folds = pd.read_csv(PHASE0_ARTIFACTS / "outer_folds.csv", dtype={"image_id": str})
    inner = pd.read_csv(PHASE0_ARTIFACTS / "inner_splits.csv", dtype={"image_id": str})
    labels = cohort.set_index("image_id")
    groups = dict(zip(cohort["image_id"], cohort["analysis_group_id"]))
    return cohort, folds, inner, labels, groups


def split_ids_for(fold, n, seed, folds, inner):
    subset = pd.read_csv(PHASE0_ARTIFACTS / "subsets" / f"subset_f{fold}_s{seed}_n{n}.csv",
                         dtype={"image_id": str})
    fit_ids = list(subset["image_id"])
    val_ids = list(inner[(inner["test_fold"] == fold) & (inner["role"] == "inner_val")]["image_id"])
    test_ids = list(folds[folds["test_fold"] == fold]["image_id"])
    return fit_ids, sorted(val_ids), sorted(test_ids)


def manifest_frame(phase_dir):
    path = Path(phase_dir) / "run_manifest.csv"
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame(columns=["run_id", "status", "finished_utc", "wall_seconds",
                                 "peak_gpu_bytes", "best_inner_macro_auc", "error",
                                 "definitions_sha256", "notebook_sha256", "gpu"])


def append_manifest_row(phase_dir, row):
    path = Path(phase_dir) / "run_manifest.csv"
    frame = pd.DataFrame([row])
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)


def save_run_outputs(run_dir, result, export_test):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    result["history"].to_csv(run_dir / "history.csv", index=False)
    torch.save(result["state"], run_dir / "checkpoint.pt")
    inner = result["inner"]
    np.savez_compressed(run_dir / "predictions_inner.npz",
                        image_ids=np.array(inner["image_ids"]),
                        labels=inner["labels"], probs=inner["probs"])
    if export_test and "test" in result:
        test = result["test"]
        np.savez_compressed(run_dir / "predictions_test.npz",
                            image_ids=np.array(test["image_ids"]),
                            labels=test["labels"], probs=test["probs"])
    write_json(run_dir / "thresholds.json", result["thresholds"])
    write_json(run_dir / "meta.json", result["meta"])


def execute_pending_runs(phase_dir, specs, splits_ctx, notebook_name, export_test=True):
    """Returns 'complete' or 'pending'. Raises PhaseStop when a run fails."""
    phase_dir = Path(phase_dir)
    phase_dir.mkdir(parents=True, exist_ok=True)
    manifest = manifest_frame(phase_dir)
    done = set(manifest.loc[manifest["status"] == "success", "run_id"])
    pending = [s for s in specs if s["run_id"] not in done]
    if not pending:
        print(f"all {len(specs)} runs already have a successful manifest row")
        return "complete"
    if not (TORCH_AVAILABLE and torch.cuda.is_available()):
        print(f"PENDING: {len(pending)} of {len(specs)} runs need a CUDA device. "
              "Run this notebook on the GPU pod.")
        return "pending"
    enable_torch_determinism()
    notebook_sha = sha256_file(PROJECT_ROOT / notebook_name)
    defs_sha = definitions_sha256()
    gpu_name = torch.cuda.get_device_name(0)
    cohort, folds, inner, labels, groups = splits_ctx
    failures = []
    for k, spec in enumerate(pending):
        print(f"[{k + 1}/{len(pending)}] {spec['run_id']}", flush=True)
        try:
            fit_ids, val_ids, test_ids = split_ids_for(spec["fold"], spec["n"], spec["seed"],
                                                       folds, inner)
            splits = {"fit_ids": fit_ids, "val_ids": val_ids,
                      "test_ids": test_ids if export_test else None, "labels": labels}
            result = train_run(spec, splits)
            save_run_outputs(phase_dir / "runs" / spec["run_id"], result, export_test)
            append_manifest_row(phase_dir, {
                "run_id": spec["run_id"], "status": "success",
                "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                "wall_seconds": result["meta"]["wall_seconds"],
                "peak_gpu_bytes": result["meta"]["peak_gpu_bytes"],
                "best_inner_macro_auc": result["meta"]["best_inner_macro_auc"],
                "error": "", "definitions_sha256": defs_sha,
                "notebook_sha256": notebook_sha, "gpu": gpu_name})
        except Exception as exc:  # noqa: BLE001 - every failure lands in the manifest
            append_manifest_row(phase_dir, {
                "run_id": spec["run_id"], "status": "failed",
                "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                "wall_seconds": "", "peak_gpu_bytes": "", "best_inner_macro_auc": "",
                "error": f"{type(exc).__name__}: {exc}",
                "definitions_sha256": defs_sha, "notebook_sha256": notebook_sha,
                "gpu": gpu_name})
            failures.append((spec["run_id"], str(exc)))
    if failures:
        raise PhaseStop(f"{len(failures)} runs failed: {failures[:5]}. "
                        "Rerun the notebook to retry them with the same configuration.")
    return "complete"


def phase2_run_specs():
    specs = []
    for cell in ["A", "B", "C", "D", "E", "F"]:
        for fold in range(1, N_FOLDS + 1):
            specs.append({"run_id": f"p2_{cell}_f{fold}_s42_n1044", "cell": cell,
                          "fold": fold, "seed": 42, "n": 1044})
    for fold in range(1, N_FOLDS + 1):
        specs.append({"run_id": f"p2_probe_f{fold}_s42_n1044", "cell": "probe",
                      "fold": fold, "seed": 42, "n": 1044})
    return specs


def phase3_gate_specs():
    return [{"run_id": f"p3_{cell}_f{fold}_s{seed}_n50", "cell": cell, "fold": fold,
             "seed": seed, "n": 50}
            for cell in ["A", "B", "C", "D"]
            for fold in range(1, N_FOLDS + 1)
            for seed in [42, 43, 44]]


def phase3_continuation_specs():
    combos = [(100, 42), (100, 43), (251, 42), (501, 42)]
    return [{"run_id": f"p3_{cell}_f{fold}_s{seed}_n{n}", "cell": cell, "fold": fold,
             "seed": seed, "n": n}
            for cell in ["A", "B", "C", "D"]
            for fold in range(1, N_FOLDS + 1)
            for n, seed in combos]


def kernel_study_specs():
    return [{"run_id": f"p1_kernel{k}_n{n}", "cell": "B", "fold": 1, "seed": 42,
             "n": n, "dka_large": k}
            for k in KERNEL_STUDY_LARGE for n in [50, 1044]]


In [ ]:
# ---------------------------------------------------------------------------
# Paired group bootstrap and the registered contrasts (protocol sections 6, 7).
# ---------------------------------------------------------------------------

def load_test_predictions(phase_dir, run_id):
    z = np.load(Path(phase_dir) / "runs" / run_id / "predictions_test.npz", allow_pickle=False)
    return {"image_ids": [str(i) for i in z["image_ids"]],
            "labels": z["labels"], "probs": z["probs"]}


def draw_replicates(fold_group_lists, replicates, seed, fold_labels):
    """Group bootstrap indices per (replicate, fold). A replicate redraws until
    every fold keeps at least one class with both label values. Returns the
    index arrays, the per-fold included-class masks, and the discard count."""
    rng = np.random.default_rng(seed)
    out_indices = []
    out_masks = []
    discarded = 0
    n_classes = len(TARGETS)
    while len(out_indices) < replicates:
        rep_idx, rep_masks, valid = {}, {}, True
        for fold, group_list in fold_group_lists.items():
            n_groups = len(group_list)
            draw = rng.integers(0, n_groups, size=n_groups)
            idx = np.concatenate([group_list[g] for g in draw])
            labels = fold_labels[fold][idx]
            mask = np.array([0 < labels[:, k].sum() < len(idx) for k in range(n_classes)])
            if not mask.any():
                valid = False
            rep_idx[fold] = idx
            rep_masks[fold] = mask
        if valid:
            out_indices.append(rep_idx)
            out_masks.append(rep_masks)
        else:
            discarded += 1
    return out_indices, out_masks, discarded


def fold_group_index_lists(image_ids, groups):
    """Maps each analysis group of one fold's test set to its image positions."""
    by_group = {}
    for pos, image_id in enumerate(image_ids):
        by_group.setdefault(groups[image_id], []).append(pos)
    return [np.array(v, dtype=np.int64) for _, v in sorted(by_group.items())]


def contrast_point_estimates(cell_fold_auc):
    a, b, c, d = (np.array([cell_fold_auc[x][f] for f in range(1, N_FOLDS + 1)])
                  for x in ["A", "B", "C", "D"])
    return {
        "C1": float(np.mean(0.5 * ((c - a) + (d - b)))),
        "C2": float(np.mean(0.5 * ((b - a) + (d - c)))),
        "C3": float(np.mean((d - c) - (b - a))),
    }


def bootstrap_summary(name, t_hat, t_boot):
    t_boot = np.asarray(t_boot, dtype=np.float64)
    lo, hi = np.quantile(t_boot, [0.00625, 0.99375])
    d = t_boot - t_hat
    p_raw = (1 + int(np.sum(np.abs(d) >= abs(t_hat)))) / (len(t_boot) + 1)
    return {"contrast": name, "estimate": t_hat, "ci_low": float(lo), "ci_high": float(hi),
            "p_raw": float(p_raw), "p_adjusted": float(min(1.0, 4 * p_raw)),
            "replicates": int(len(t_boot))}


def dka_effect(cells_auc):
    return 0.5 * ((cells_auc["B"] - cells_auc["A"]) + (cells_auc["D"] - cells_auc["C"]))


def c4_slope(effects_by_n):
    """effects_by_n: {n: fold-averaged effect}. Unweighted OLS on log10(n)."""
    ns = sorted(effects_by_n)
    x = np.log10(np.array(ns, dtype=np.float64))
    y = np.array([effects_by_n[n] for n in ns], dtype=np.float64)
    slope, _ = np.polyfit(x, y, 1)
    return float(slope)


def pooled_sd(per_fold_sd, seeds_minus_one, dof):
    s = np.asarray(per_fold_sd, dtype=np.float64)
    return float(np.sqrt(np.sum(seeds_minus_one * s ** 2) / dof))


# Parallel replicate execution. draw_replicates stays serial and untouched;
# workers only evaluate the replicate function on already-drawn indices, and
# the parent merges block results in replicate order, so the output is
# bit-identical to the serial loop. Workers start with fork; any failure
# falls back to the serial loop.
import multiprocessing as _mp

_BOOT_CTX = {}


def _replicate_block_worker(block):
    start, end = block
    fn = _BOOT_CTX["fn"]
    return [fn(r) for r in range(start, end)]


def run_replicates(fn, n_replicates, workers=None, parallel=True):
    if not parallel:
        return [fn(r) for r in range(n_replicates)]
    workers = workers or os.cpu_count() or 1
    if workers <= 1 or n_replicates < 2 * workers:
        return [fn(r) for r in range(n_replicates)]
    try:
        ctx = _mp.get_context("fork")
    except ValueError:
        print("fork is unavailable here: running the serial bootstrap")
        return [fn(r) for r in range(n_replicates)]
    _BOOT_CTX["fn"] = fn
    bounds = np.linspace(0, n_replicates, workers * 4 + 1).astype(int)
    blocks = [(int(a), int(b)) for a, b in zip(bounds[:-1], bounds[1:]) if b > a]
    try:
        with ctx.Pool(processes=workers) as pool:
            chunks = pool.map(_replicate_block_worker, blocks)
        out = [item for chunk in chunks for item in chunk]
        if len(out) != n_replicates:
            raise RuntimeError(f"parallel bootstrap returned {len(out)} results")
        return out
    except Exception as exc:
        print(f"parallel bootstrap failed ({type(exc).__name__}: {exc}); running serial")
        return [fn(r) for r in range(n_replicates)]
    finally:
        _BOOT_CTX.pop("fn", None)
